# GwenLand E2E — bench + train, on Kaggle

Two phases, both run end to end against real code:

| Phase | What runs | Hardware |
|---|---|---|
| **1 — bench** | `glbench run` on Qwen2.5-0.5B | GPU (`glcuda`) if present, else CPU (`glproc`) |
| **2 — train** | `glbench train` — one Linear layer + LoRA | **CPU only** |

### Phase 2 is CPU-only, and that is not a limitation to work around

`gltrain` (Stummañ) has exactly two backends: `BEGlProc` (CPU) and `BESisd`.
There is no CUDA backend in the training crate. A T4 changes nothing about
Phase 2, so this notebook does not pretend otherwise.

### What Phase 2 actually trains

**One 256×256 Linear layer with a LoRA adapter at rank 8.** Not a language
model. 4,096 trainable parameters against a 65,536-parameter frozen base
(6.25%). Success is **loss slope < 0** — the training loop works end to end.

`synthetic_regression`'s targets are not normalised, so the loss is large in
absolute terms (order 10¹–10²) and *should* be. Do not read "loss 95" as
failure; read the slope.

### Before you run

1. **Internet must be ON** (Settings → Internet). Kaggle is offline by default.
2. GPU is optional. Phase 1 falls back to CPU automatically.
3. The build is the slow part — roughly **8–15 minutes** from cold. That is an
   estimate, not a measurement; nobody has timed this build on Kaggle.

## 0 — Configuration

Everything you might need to change is here.

In [ ]:
# ---- edit these if needed -------------------------------------------------
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BRANCH   = "glbench-v3-wave2"   # v3 lives here, NOT on main
GH_TOKEN = ""                   # only if the repo is private

# Phase 1 model. The Qwen official GGUF repo.
MODEL_REPO = "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/main"
MODEL_FILE = "qwen2.5-0.5b-instruct-q4_k_m.gguf"
# Fallbacks, tried in order if the first 404s.
MODEL_FALLBACKS = ["qwen2.5-0.5b-instruct-q8_0.gguf", "qwen2.5-0.5b-instruct-q5_k_m.gguf"]

# Phase 1 workload.
#
# There is deliberately no PROMPT. glbench's own default_prompt() repeats a
# base sentence 8x so prefill has real work to measure; a short prompt makes
# prefill tok/s a measure of launch overhead instead. Overriding it once
# already produced a meaningless 1835 tok/s over ~11 tokens.
GEN_TOKENS  = 128
WARMUP      = 1
ITERS       = 3

# Phase 2 training shape
D_IN, D_OUT, RANK = 256, 256, 8
SAMPLES, EPOCHS   = 32, 10
# --------------------------------------------------------------------------

import os, sys, shutil, subprocess, json, time, urllib.request

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
os.makedirs(WORK, exist_ok=True)
REPO_DIR = os.path.join(WORK, "gwenland-ai")

def have_internet(timeout=6):
    try:
        urllib.request.urlopen("https://github.com", timeout=timeout)
        return True
    except Exception:
        return False

print(f"working dir : {WORK}")
print(f"internet    : {'YES' if have_internet() else 'NO  <-- enable it in Settings -> Internet'}")
print(f"branch      : {BRANCH}")

try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True, timeout=30)
    HAS_GPU = out.returncode == 0 and out.stdout.strip() != ""
    print(f"gpu         : {out.stdout.strip() if HAS_GPU else 'none'}")
except Exception:
    HAS_GPU = False
    print("gpu         : none (nvidia-smi not found)")

## 1 — Kaggle path shim

Makes the `/content` paths the other notebooks in `notebooks/` use work unchanged here.

In [ ]:
if os.path.isdir("/kaggle/working") and not os.path.exists("/content"):
    subprocess.run(["ln", "-sfn", "/kaggle/working", "/content"], check=False)
    print("linked /content -> /kaggle/working")
else:
    print("no shim needed")

## 2 — Rust toolchain

`--profile minimal`: no docs, no clippy. This build needs neither.

In [ ]:
if shutil.which("cargo") is None:
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable --profile minimal
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
!rustc --version && cargo --version

## 3 — Clone

In [ ]:
url = REPO_URL
if GH_TOKEN:
    url = REPO_URL.replace("https://", f"https://{GH_TOKEN}@")

if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)

r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, REPO_DIR],
                   capture_output=True, text=True)
if r.returncode != 0:
    print("CLONE FAILED\n" + r.stderr[-2000:])
    print("\nCheck: internet enabled? branch name right? private repo needs GH_TOKEN.")
else:
    os.chdir(REPO_DIR)
    print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
    print("crates:", ", ".join(sorted(d for d in os.listdir(".") if d.startswith("gl"))))

## 4 — Build

One command builds both phases. `--features train-bench` pulls `gltrain` in as
a path dependency.

> **`cargo build -p glbench -p gltrain` does NOT work.** `gltrain` declares its
> own `[workspace]` and is in the root `exclude` list, so `-p gltrain` from the
> repo root fails with *"package ID specification `gltrain` did not match any
> packages"*. The feature flag is the supported way in.

**Run this cell once per session.** Re-running it over an already-built
tree measures a no-op relink, not a build. That is exactly how a first run
reported "0.5 min" for a fat-LTO release build of the whole workspace. The
cell reports which of the two it actually did.

No CUDA toolkit is needed. `glcuda` ships **pre-compiled PTX**
(`glcuda/src/kernels/glcuda_sm75.ptx` — sm_75 is Turing, which is exactly the
T4) and JITs it through the driver at runtime.

In [ ]:
os.chdir(REPO_DIR)
# A build is only "cold" when there is nothing to reuse. Decided BEFORE the
# timer starts, so the number below can never be mistaken for what it isn't.
COLD_BUILD = not os.path.isdir(os.path.join(REPO_DIR, "target", "release"))
t0 = time.time()
!cargo build --release -p glbench --features train-bench 2>&1 | tail -25
BUILD_SECS = time.time() - t0

BIN = os.path.join(REPO_DIR, "target/release/glbench")
BUILD_OK = os.path.exists(BIN)
kind = "COLD, from scratch" if COLD_BUILD else "WARM - a relink, NOT a build time"
print(f"\nbuild: {'OK' if BUILD_OK else 'FAILED'} in {BUILD_SECS/60:.1f} min  [{kind}]")
if BUILD_OK:
    print(subprocess.run([BIN, "help"], capture_output=True, text=True).stdout.splitlines()[0])

## 5 — Engine order

No probing with a fake model path: glcuda availability is only settled when it actually initialises, so Phase 1 **tries each engine in turn and keeps the first that works**. A real test, not a guess.

In [ ]:
ENGINE_ORDER = (["glcuda", "glproc"] if HAS_GPU else ["glproc"])
print("will try, in order:", " -> ".join(ENGINE_ORDER))

## 6 — Model

> The file named `q4_k_m` is **not** mostly Q4_K. Measured on this model in this repo: ~67% Q5_0, ~9% Q4_K. `Q4_K_M` is a filename, not a description of the contents. Reported as-is below.

In [ ]:
os.chdir(REPO_DIR)
MODEL_PATH = None
for name in [MODEL_FILE] + MODEL_FALLBACKS:
    dest = os.path.join(REPO_DIR, name)
    if os.path.exists(dest) and os.path.getsize(dest) > 10_000_000:
        MODEL_PATH = dest; break
    print(f"fetching {name} ...")
    rc = os.system(f'wget -q -O "{dest}" "{MODEL_REPO}/{name}"')
    if rc == 0 and os.path.getsize(dest) > 10_000_000:
        MODEL_PATH = dest; break
    if os.path.exists(dest):
        os.remove(dest)
    print(f"  not available, trying next")

if MODEL_PATH:
    print(f"\nmodel: {os.path.basename(MODEL_PATH)}  ({os.path.getsize(MODEL_PATH)/2**20:.0f} MiB)")
else:
    print("\nNO MODEL — phase 1 will be skipped")

## 7 — Phase 1: inference benchmark

In [ ]:
PHASE1 = {"ok": False, "reason": None}
ENGINE = None
p1_json = os.path.join(REPO_DIR, "phase1.json")

if not BUILD_OK:
    PHASE1["reason"] = "build failed"
elif not MODEL_PATH:
    PHASE1["reason"] = "no model"
else:
    # Try each candidate, keep the first that works. glcuda's availability
    # is only settled when it actually initialises -- build_engine() returns
    # Ok for it unconditionally -- so attempting the real run IS the probe.
    #
    # No --prompt: glbench uses default_prompt(), sized so prefill has
    # enough tokens to mean something.
    for engine in ENGINE_ORDER:
        cmd = [BIN, "run", "--engine", engine, "--model", MODEL_PATH,
               "--tokens", str(GEN_TOKENS),
               "--warmup", str(WARMUP), "--iters", str(ITERS),
               "--out", p1_json]
        print(f"--- trying --engine {engine} ---")
        try:
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
            if r.returncode == 0:
                print(r.stdout[-3000:])
                PHASE1["ok"] = True
                ENGINE = engine
                break
            PHASE1["reason"] = f"{engine}: exit {r.returncode}"
            tail = (r.stderr.strip().splitlines() or ["<no stderr>"])[-1]
            print(f"    failed (exit {r.returncode}): {tail[:200]}")
        except subprocess.TimeoutExpired:
            PHASE1["reason"] = f"{engine}: timed out"
            print("    timed out")
        except Exception as e:
            PHASE1["reason"] = f"{engine}: {type(e).__name__}: {e}"
            print(f"    {type(e).__name__}: {e}")

if not PHASE1["ok"]:
    print(f"\nPHASE 1 NOT OK: {PHASE1['reason']}")

### Phase 1 numbers

Read from the JSON archive rather than parsed out of stdout — the archive is the stable interface.

In [ ]:
if PHASE1["ok"] and os.path.exists(p1_json):
    s = json.load(open(p1_json, encoding="utf-8"))
    its = s["measurements"]["iterations"]
    dec = [i["generated_tokens"] / (i["decode_ms"]/1000) for i in its if i["decode_ms"] > 0]
    pre = [i["prompt_tokens"]   / (i["prefill_ms"]/1000) for i in its if i["prefill_ms"] > 0]
    cold = s["measurements"].get("cold") or []

    PHASE1.update(
        engine=s["engine"]["name"],
        quant=s["engine"].get("quantization"),
        arch=s["engine"].get("model_arch"),
        decode_tps=sum(dec)/len(dec) if dec else None,
        prefill_tps=sum(pre)/len(pre) if pre else None,
        cold_ms=cold[0]["total_ms"] if cold else None,
        iters=len(its),
        prompt_tokens=its[0]["prompt_tokens"] if its else 0,
        schema=s["metadata"]["schema_version"],
    )
    print(f"engine      : {PHASE1['engine']}   arch {PHASE1['arch']}   quant {PHASE1['quant']}")
    ptok = PHASE1["prompt_tokens"]
    warn = "" if ptok >= 100 else "   <-- too few tokens to mean much"
    print(f"prefill     : {PHASE1['prefill_tps']:.1f} tok/s over {ptok} prompt tokens{warn}")
    print(f"decode      : {PHASE1['decode_tps']:.1f} tok/s   (mean of {PHASE1['iters']} warm iters)")
    if PHASE1["cold_ms"]:
        print(f"cold start  : {PHASE1['cold_ms']/1000:.2f} s  (reported separately, never folded into the warm mean)")
    print(f"archive     : schema v{PHASE1['schema']}, digest verified on read")
else:
    print("skipped")

## 8 — Phase 2: training

`glbench train` builds a `Trainer`, installs an observer, and calls gltrain's
own `Trainer::train`. glbench never drives the loop — it watches one.

There is deliberately **no `--model` or `--dataset`**: gltrain generates its
frozen base weight from `--seed` and builds the dataset in memory. The shape
and seed flags below fully determine the run.

In [ ]:
PHASE2 = {"ok": False, "reason": None}
p2_json = os.path.join(REPO_DIR, "phase2.json")

if not BUILD_OK:
    PHASE2["reason"] = "build failed"
else:
    cmd = [BIN, "train",
           "--d-in", str(D_IN), "--d-out", str(D_OUT), "--rank", str(RANK),
           "--samples", str(SAMPLES), "--epochs", str(EPOCHS),
           "--label", "kaggle-e2e", "--out", p2_json]
    print(" ".join(cmd), "\n")
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
        print(r.stdout[-4000:])
        if r.returncode != 0:
            PHASE2["reason"] = f"exit {r.returncode}"
            print("--- STDERR ---\n" + r.stderr[-3000:])
        else:
            PHASE2["ok"] = True
    except subprocess.TimeoutExpired:
        PHASE2["reason"] = "timed out"
    except Exception as e:
        PHASE2["reason"] = f"{type(e).__name__}: {e}"

if not PHASE2["ok"]:
    print(f"\nPHASE 2 NOT OK: {PHASE2['reason']}")

### Phase 2 numbers

Success is **slope < 0**. The absolute loss is large because the synthetic targets are unnormalised — that is the dataset, not a bug.

In [ ]:
if PHASE2["ok"] and os.path.exists(p2_json):
    s = json.load(open(p2_json, encoding="utf-8"))
    t = s["training"]; c = t["convergence"]; a = t["adapter"]; att = t["attribution"]

    PHASE2.update(
        first=c["first_loss"], final=c["final_loss"], best=c["best_loss"],
        slope=c["slope_per_step"], steps=t["steps_observed"],
        trainable=a["trainable_parameters"], base=a["base_parameters"],
        ratio=a["parameter_ratio"], ms_per_step=att["mean_step_ms"],
        descended=c["slope_per_step"] < 0,
    )
    print(f"adapter     : {a['kind']} rank {a['rank']} over {a['d_in']}x{a['d_out']}")
    print(f"              {a['trainable_parameters']} trainable / {a['base_parameters']} base "
          f"({a['parameter_ratio']*100:.2f}%)")
    print(f"steps       : {t['steps_observed']} observed, {t['steps_archived']} archived "
          f"(sample N={t['step_sample_n']})")
    print(f"loss        : {c['first_loss']:.4f} -> {c['final_loss']:.4f}   best {c['best_loss']:.4f}")
    print(f"slope       : {c['slope_per_step']:+.4e} per step   "
          f"{'DESCENDING' if PHASE2['descended'] else 'NOT DESCENDING'}")
    print(f"step time   : {att['mean_step_ms']:.4f} ms mean  "
          f"(fwd {att['forward_share']*100:.0f}% / bwd {att['backward_share']*100:.0f}% "
          f"/ opt {att['optimizer_share']*100:.0f}%)")
else:
    print("skipped")

### Loss curve

Rendered by `glbench`'s own ASCII plotter, via the Markdown export.

In [ ]:
if PHASE2["ok"]:
    r = subprocess.run([BIN, "export", p2_json, "--format", "md"],
                       capture_output=True, text=True)
    md_out = r.stdout
    if "### Loss curve" in md_out:
        block = md_out.split("### Loss curve", 1)[1].split("###", 1)[0]
        print(block.replace("```text", "").replace("```", "").strip())
    else:
        print("(no curve section — training may have archived no steps)")

## 9 — Report

In [ ]:
line = "=" * 66
print(line)
print("GwenLand E2E — combined report")
print(line)
print(f"repo      : {REPO_URL.rsplit('/', 2)[-2]}/{REPO_URL.rsplit('/', 1)[-1].replace('.git','')} @ {BRANCH}")
bkind = "cold" if COLD_BUILD else "WARM - not a build time"
print(f"build     : {'OK' if BUILD_OK else 'FAILED'}  ({BUILD_SECS/60:.1f} min, {bkind})")
print(f"gpu       : {'yes' if HAS_GPU else 'no'}")
print()

# ---- phase 1
if PHASE1["ok"]:
    print(f"PHASE 1  PASS   engine {PHASE1['engine']} ({PHASE1['quant']})")
    print(f"                prefill {PHASE1['prefill_tps']:.1f} tok/s "
          f"over {PHASE1['prompt_tokens']} prompt tokens | "
          f"decode {PHASE1['decode_tps']:.1f} tok/s")
else:
    print(f"PHASE 1  FAIL   {PHASE1['reason']}")

# ---- phase 2: the criterion is the slope, not the magnitude
if PHASE2["ok"]:
    verdict = "PASS" if PHASE2["descended"] else "FAIL"
    print(f"PHASE 2  {verdict}   one {D_IN}x{D_OUT} Linear + LoRA r{RANK}, "
          f"{PHASE2['trainable']} trainable params")
    print(f"                loss {PHASE2['first']:.3f} -> {PHASE2['final']:.3f}, "
          f"slope {PHASE2['slope']:+.3e}/step")
    print(f"                {PHASE2['ms_per_step']:.3f} ms/step on CPU")
else:
    print(f"PHASE 2  FAIL   {PHASE2['reason']}")

print(line)
both = PHASE1.get("ok") and PHASE2.get("ok") and PHASE2.get("descended")
print("RESULT:", "both phases passed" if both else "see failures above")
print(line)
print()
print("Caveats that belong with these numbers:")
print(" - Phase 2 ran on CPU. gltrain has no CUDA backend; a GPU changes nothing there.")
print(" - Phase 2 trains ONE Linear layer, not a language model. The claim is that")
print("   the training loop works end to end, nothing more.")
print(" - Phase 2's loss is large in absolute terms because synthetic_regression's")
print("   targets are unnormalised. The success criterion is the slope.")
if PHASE1.get("ok") and PHASE1.get("prompt_tokens", 0) < 100:
    print(" - Prefill covered fewer than 100 prompt tokens. At that size the number")
    print("   reflects launch overhead more than prefill throughput.")
if not COLD_BUILD:
    print(" - The build was WARM. That figure is a relink, not a build time.")
print(" - Phase 1 is one machine, one session. Kaggle instances vary; do not compare")
print("   these tok/s against numbers from another box.")